# 🔑 Solutions — Functions — advanced concepts

One worked answer per exercise. There is usually more than one correct
way to write these — if your self-check printed ✅, yours is right too.

Read this *after* an attempt: seeing the answer teaches far less than
getting stuck did.

## Our data

In [ ]:
# The genetic code: every three-letter codon -> its amino acid (one letter).
codon_table = {
    "TTT": "F", "TTC": "F", "TTA": "L", "TTG": "L",
    "CTT": "L", "CTC": "L", "CTA": "L", "CTG": "L",
    "ATT": "I", "ATC": "I", "ATA": "I", "ATG": "M",
    "GTT": "V", "GTC": "V", "GTA": "V", "GTG": "V",
    "TCT": "S", "TCC": "S", "TCA": "S", "TCG": "S",
    "CCT": "P", "CCC": "P", "CCA": "P", "CCG": "P",
    "ACT": "T", "ACC": "T", "ACA": "T", "ACG": "T",
    "GCT": "A", "GCC": "A", "GCA": "A", "GCG": "A",
    "TAT": "Y", "TAC": "Y", "TAA": "*", "TAG": "*",
    "CAT": "H", "CAC": "H", "CAA": "Q", "CAG": "Q",
    "AAT": "N", "AAC": "N", "AAA": "K", "AAG": "K",
    "GAT": "D", "GAC": "D", "GAA": "E", "GAG": "E",
    "TGT": "C", "TGC": "C", "TGA": "*", "TGG": "W",
    "CGT": "R", "CGC": "R", "CGA": "R", "CGG": "R",
    "AGT": "S", "AGC": "S", "AGA": "R", "AGG": "R",
    "GGT": "G", "GGC": "G", "GGA": "G", "GGG": "G",
}
stop_codons = {"TAA", "TAG", "TGA"}

# The same short gene, sampled from two related species.
species_a = "ATGAAAGGTTTTCTGGAATATGCAGGCTAA"   # the demos use this
species_b = "ATGAAAGGCTTCCTGGAGTATGCAAGCTAA"   # your exercises use this

print("species A:", species_a)
print("species B:", species_b)


# our translate() from the basics notebook - we'll build on it here
def translate(seq, frame=0):
    protein = ""
    for i in range(frame, len(seq) - 2, 3):
        codon = seq[i:i + 3]
        if codon in stop_codons:
            break
        protein = protein + codon_table[codon]
    return protein

## Solution — Skill 1, 🔬 write it: document `translate`

In [ ]:
def translate(seq: str, frame: int = 0) -> str:
    """Translate DNA into protein, from frame until the first stop codon."""
    protein = ""
    for i in range(frame, len(seq) - 2, 3):
        codon = seq[i:i + 3]
        if codon in stop_codons:
            break
        protein = protein + codon_table[codon]
    return protein

help(translate)
print(translate(species_b))             # MKGFLEYAS

## Solution — Skill 2, 🐞 broken: too many names

In [ ]:
# gc_at from the demo, so this notebook runs on its own
def gc_at(seq: str) -> tuple[float, float]:
    """Return the GC% and the AT% of seq, both to 1 dp."""
    gc = 0
    at = 0
    for base in seq:
        if base in "GC":
            gc = gc + 1
        elif base in "AT":
            at = at + 1
    return round(100 * gc / len(seq), 1), round(100 * at / len(seq), 1)

gc, at = gc_at(species_b)               # two values back -> two names
print("species B GC%:", gc)             # 43.3
print("species B AT%:", at)             # 56.7

# if you also want the length, ask for it separately:
length = len(species_b)

## Solution — Skill 2, 🔬 write it: `compare`

In [ ]:
def compare(seq_a: str, seq_b: str) -> tuple[int, int]:
    """Return how many DNA positions, and how many protein positions, differ."""
    dna_diffs = 0
    for i in range(len(seq_a)):
        if seq_a[i] != seq_b[i]:
            dna_diffs = dna_diffs + 1

    protein_a = translate(seq_a)
    protein_b = translate(seq_b)
    protein_diffs = 0
    for i in range(len(protein_a)):
        if protein_a[i] != protein_b[i]:
            protein_diffs = protein_diffs + 1

    return dna_diffs, protein_diffs

dna_diffs, protein_diffs = compare(species_a, species_b)
print("DNA differences:    ", dna_diffs)       # 4
print("protein differences:", protein_diffs)   # 1

## Solution — Skill 3, 🔬 write it: a bigger summary

In [ ]:
def analyse(seq: str) -> dict:
    """Return the length, GC%, AT% and protein of a DNA sequence."""
    gc, at = gc_at(seq)                 # Skill 2 inside Skill 3: unpack, then label
    return {
        "length": len(seq),
        "gc": gc,
        "at": at,
        "protein": translate(seq),
    }

result_a = analyse(species_a)
result_b = analyse(species_b)
print("GC%:    ", result_b["gc"])       # 43.3
print("protein:", result_b["protein"])  # MKGFLEYAS

drift = "richer"                        # 36.7 -> 43.3: species B is GC-richer

## Solution — Skill 4, 🔬 write it: the position in the codon

In [ ]:
# classify_mutation from the demo
def classify_mutation(seq: str, position: int, new_base: str) -> str:
    """Swap one base and report whether the protein changes."""
    protein_before = translate(seq)
    mutated = seq[:position] + new_base + seq[position + 1:]
    protein_after = translate(mutated)
    if protein_after == protein_before:
        return "synonymous"
    return "missense"

silent_third = 0
for position in range(5, 27, 3):                # 3rd bases: 5, 8, 11 ... 26
    for new_base in "ACGT":
        if new_base != species_b[position]:     # skip the base that is already there
            if classify_mutation(species_b, position, new_base) == "synonymous":
                silent_third = silent_third + 1

silent_first = 0
for position in range(3, 27, 3):                # 1st bases: 3, 6, 9 ... 24
    for new_base in "ACGT":
        if new_base != species_b[position]:
            if classify_mutation(species_b, position, new_base) == "synonymous":
                silent_first = silent_first + 1

print("silent at the 3rd base:", silent_third, "of 24")   # 14
print("silent at the 1st base:", silent_first, "of 24")   # 1